# A Multigroup 2D Glovebox

In this example we will explore OpenSn's discrete ordinates solver for computing response functions in two Helium-3 detectors of a mockup 2D glovebox.

This is a complete simulation transport example. Each aspect of this example can be broken as follows:
- [Prerequisites](#prerequisites)
- [Mesh](#mesh)
- [Geometry](#Geometry)
    - [Cross Sections](#cross-sections)
- [Solver](#Solver)
    - [Physics Options](#physics-options)
    - [Angular Quadrature](#angular-quadrature)
    - [Group Structure](#group-structure)
    - [Source Definition](#source-definition)
    - [Discrete Ordinates Problem](#discrete-ordinates-problem)
    - [Execute](#execute)
- [Post Processing](#post-processing)
    - [Volumetric Field Function](#volumetric-field-function)
    - [Compute Leakage](#compute-leakage)
    - [Compute Balance](#compute-balance)

---
## Prerequisites

Before running this example, make sure that the **Python module of OpenSn** was installed.

### Converting and Running this Notebook from the Terminal
To run this notebook from the terminal, simply type:

`jupyter nbconvert --to python --execute glovebox.ipynb`.

To run this notebook in parallel (for example, using 4 processes), simply type:

`mpiexec -n 4 jupyter nbconvert --to python --execute glovebox.ipynb`.

In [ ]:
from mpi4py import MPI
size = MPI.COMM_WORLD.size
rank = MPI.COMM_WORLD.rank
barrier = MPI.COMM_WORLD.barrier

if rank == 0:
    print(f"Running the first LBS Adjoint example with {size} MPI processors.")

### Import Requirements

Import required classes and functions from the Python interface of OpenSn. Make sure that the path
to PyOpenSn is appended to Python's PATH.

In [ ]:
import os
import sys
import math
import numpy as np

# assuming that the execute dir is the notebook dir
# this line is not necessary when PyOpenSn is installed using pip
sys.path.append("../../..")

from pyopensn.mesh import FromFileMeshGenerator, PETScGraphPartitioner
from pyopensn.xs import MultiGroupXS
from pyopensn.source import VolumetricSource
from pyopensn.aquad import GLCProductQuadrature2DXY
from pyopensn.solver import DiscreteOrdinatesProblem, SteadyStateSourceSolver
from pyopensn.response import ResponseEvaluator
from pyopensn.math import Vector3, VectorSpatialFunction
from pyopensn.fieldfunc import FieldFunctionInterpolationVolume, \
                               FieldFunctionInterpolationLine, \
                               FieldFunctionGridBased
from pyopensn.logvol import RPPLogicalVolume
from pyopensn.context import UseColor, Finalize


## Geometry

![Glovebox](images/glovebox.png)

---
## Mesh
Here, we will be importing an unstructured mesh generated in the open-source meshing utility Gmsh.

In [ ]:
meshgen = FromFileMeshGenerator(
    filename="glovebox.msh",
    partitioner=PETScGraphPartitioner(type='parmetis'),
)
grid = meshgen.Execute()

![GloveboxMesh](images/gloveboxmesh.png)

![DetMesh](images/detmesh.png)

![SrcMesh](images/srcmesh.png)

### Cross Sections
In this problem we are running a multigroup problem for the absoprtion reaction rate in a He-3 glovebox. This begins with importing our multigroup cross sections. In this case we will be using the WIMS69 group structure for 5 materials:
- Air
- SS-316
- HDPE
- He-3
- Cf-252

Since we are importing a Gmsh mesh, materials IDs are assigned to the corresponding physcial group ID. Look at [glovebox.msh](./glovebox.msh) under `PhysicalNames`.

In [ ]:
xs_dir = "WIMS69"
xs_air = MultiGroupXS()
xs_air.LoadFromOpenSn(xs_dir+"/Air.cxs")

xs_steel = MultiGroupXS()
xs_steel.LoadFromOpenSn(xs_dir+"/SS_316.cxs")

xs_hdpe = MultiGroupXS()
xs_hdpe.LoadFromOpenSn(xs_dir+"/HDPE.cxs")

xs_he3 = MultiGroupXS()
xs_he3.LoadFromOpenSn(xs_dir+"/He3.cxs")

xs_cf = MultiGroupXS()
xs_cf.LoadFromOpenSn(xs_dir+"/Cf252.cxs")

xsecs = [{"block_ids": [1], "xs": xs_air},
         {"block_ids": [2], "xs": xs_steel},
         {"block_ids": [3], "xs": xs_hdpe},
         {"block_ids": [4], "xs": xs_he3},
         {"block_ids": [5], "xs": xs_cf}]

---
## **Solver**

### Physics Options 

In [ ]:
phys_opts = {
    "boundary_conditions": [
        {"name": "xmin", "type": "reflecting"},
        {"name": "xmax", "type": "reflecting"},
        {"name": "ymin", "type": "reflecting"},
        {"name": "ymax", "type": "reflecting"},
        {"name": "zmin", "type": "vacuum"},
        {"name": "zmax", "type": "vacuum"}
    ],
    "save_angular_flux": True,
}

### Angular Quadrature
Since we are solving a 2D problem we will create Gauss-Legendre Product Quadrature in 2D. In this case we will use 4 polar angles and 64 azimuthal angles. With polar symmetry that leaves us with 512 angles. 

In [ ]:
pquad = GLCProductQuadrature2DXY(n_polar=2, 
                                 n_azimuthal=128,
                                 scattering_order=1)

### Group Structure

In [ ]:
num_groups = 69
groups = [1.00000E-11, 5.00000E-09, 1.00000E-08, 1.50000E-08, 2.00000E-08, 
          2.50000E-08, 3.00000E-08, 3.50000E-08, 4.20000E-08, 5.00000E-08, 
          5.80000E-08, 6.70000E-08, 8.00000E-08, 1.00000E-07, 1.40000E-07, 
          1.80000E-07, 2.20000E-07, 2.50000E-07, 2.80000E-07, 3.00000E-07, 
          3.20000E-07, 3.50000E-07, 4.00000E-07, 5.00000E-07, 6.25000E-07, 
          7.80000E-07, 8.50000E-07, 9.10000E-07, 9.50000E-07, 9.72000E-07, 
          9.96000E-07, 1.02000E-06, 1.04500E-06, 1.07100E-06, 1.09700E-06, 
          1.12300E-06, 1.15000E-06, 1.30000E-06, 1.50000E-06, 2.10000E-06, 
          2.60000E-06, 3.30000E-06, 4.00000E-06, 9.87700E-06, 1.59680E-05, 
          2.77000E-05, 4.80520E-05, 7.55014E-05, 1.48729E-04, 3.67263E-04, 
          9.06899E-04, 1.42510E-03, 2.23945E-03, 3.51910E-03, 5.53000E-03, 
          9.11800E-03, 1.50300E-02, 2.47800E-02, 4.08500E-02, 6.73400E-02, 
          1.11000E-01, 1.83000E-01, 3.02500E-01, 5.00000E-01, 8.21000E-01, 
          1.35300E+00, 2.23100E+00, 3.67900E+00, 6.06550E+00, 1.00000E+01]
grpsets = [
    {
        "groups_from_to": (0, num_groups-1),
        "angular_quadrature": pquad,
        "inner_linear_method": "petsc_gmres",
        "l_abs_tol": 1.0e-6,
        "l_max_its": 300,
        "gmres_restart_interval": 100,
    },
]

### Source Definition

For the volumetric source we first define the source sprectrum. In this case we will assign a uniform spectrum in energy distributed along the source logical volume. Using `VolumetricSource` we assign this group-wise source definition to material ID 4. With a `VoluemtricSource` defined it is then added to the physics options. 

In [ ]:
group_width = np.flip(np.diff(groups))
dE = groups[-1] - groups[0]
dA = np.pi*0.2*0.2
Q = (group_width / dE / dA).tolist()

vol_src = VolumetricSource(block_ids=[4], group_strength=Q)
phys_opts["volumetric_sources"] = [vol_src]

### Discrete Ordinates Problem
For the problem block, we provide
+ mesh : The mesh
+ num_groups : The number of energy groups
+ groupsets : The groupsets block
+ scattering_order : The scattering order
+ xs_map : Cross section map
+ options : Physics solver options

In [ ]:
phys = DiscreteOrdinatesProblem(
    mesh=grid,
    num_groups=num_groups,
    groupsets=grpsets,
    scattering_order=1,
    xs_map=xsecs,
    options=phys_opts
)

### Execute
We then create the physics solver, initialize it, and execute it.

In [ ]:
ss_solver = SteadyStateSourceSolver(problem=phys)
ss_solver.Initialize()
ss_solver.Execute()

## Post Processing
### Volumetric Field Function

With the solver executed, we now create a `FieldFunction`. In OpenSn we define a `FieldFunction` for the response we will like to calculate. In this case we are looking to compute the total $He^3$ absorption reaction rate in our glovebox: 

$$
\text{QoI} = \text{RR}^g(\mathcal{D}) = \sum^G_g \, \int_\mathcal{D} d^3r \, \int_{(4\pi)} d\Omega \, \sigma^g_{det}(\vec{r}) \psi^g(\vec{r},\vec\Omega)  = \sum^G_g \, \int_\mathcal{D} d^3r \, \sigma^g_{det}(\vec{r}) \phi^g(\vec{r}) 
$$

Thus, in OpenSn we will generate a scalar field function using `GetScalarFieldFunctionList` with a `sum` over the RoI(s). In this case we have two detectors so we will create two logical volumes as our RoI's.

In [ ]:
det1 = [7.0, 10.0, 0.0]
det2 = [47.0, 10.0, 0.0]
dets = [det1, det2]
det_logvols = []
for det in dets:
    r_he3 = 1.2
    he3 = RCCLogicalVolume(
                r=r_he3, 
                x0=det[0], y0=det[1], 
                z0=-2.0, vz= 2.0
    )
    det_logvols.append(he3)

At each group the solution $\phi^g(\vec{r})$ is multiplied by the detector response at that group $\sigma^{He3,g}_{a}$. 

In [ ]:
fflist = phys.GetScalarFieldFunctionList(only_scalar_flux=False)
sig_a = np.array(xs_he3.sigma_a)

fields = []
flux = []
resp = []
for d,det in enumerate(det_logvols):
    flux.append([])
    resp.append([])
    for g in range(num_groups):
        ffi = FieldFunctionInterpolationVolume()
        ffi.SetOperationType("sum")
        ffi.SetLogicalVolume(det)
        ffi.AddFieldFunction(fflist[g][0])
        ffi.Initialize()
        ffi.Execute()
        phi_g = ffi.GetValue()

        flux[d].append(phi_g)
        resp[d].append(sig_a[g]*phi_g)

        if d == 0: fields.append(fflist[g][0])

if rank == 0:
    print("Total Flux : ", np.sum(flux))
    print("Total Response : ", np.sum(resp))

Total Flux :  192.88363435736423 \
Total Response :  14.390180370164801

In [ ]:
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(2,figsize=(16,9))

if rank == 0:
    erg = np.flip(groups[1:])
    for d in range(len(det_logvols)):
        ax1.loglog(erg, flux[d], "--+", label="Detector"+str(d+1), drawstyle='steps')
        ax2.loglog(erg, resp[d], "--o", label="Detector"+str(d+1), drawstyle='steps')
    ax1.set_xlabel("Energy (MeV)")
    ax1.set_ylabel(r"$\phi^g(\mathcal{D})$")
    ax1.legend()
    ax1.grid()

    ax2.set_xlabel("Energy (MeV)")
    ax2.set_ylabel(r"$\text{RR}^g(\mathcal{D})$")
    ax2.legend()
    ax2.grid()

    plt.savefig("images/Flux.png")

![Flux](images/Flux.png)

With our field function defined, we can also export the multi-group scalar flux, $\phi^g$, to a .vtu file using ``ExportMultipleToPVTU``.

In [ ]:
FFGrid = FieldFunctionGridBased
FFGrid.ExportMultipleToPVTU(fields, "Flux/glovebox")

Group 3: [1.353, 2.231] MeV 

![Glovebox_G3](images/glovebox_g3.png)

Group 52: [2.2e-07, 2.5e-07] MeV 

![Glovebox_G52](images/glovebox_g52.png)

### Compute Leakage
We can simultanously compute the groupwise leakage rate at the problem boundaries, $\vec{j}^{g}|_{\Gamma_{\text{+}}}$, of the problem domain.

In [ ]:
leakage = phys.ComputeLeakage(["xmin", "xmax","ymin","ymax"])
lkg_xmin = leakage['xmin']
lkg_xmax = leakage['xmax']
lkg_ymin = leakage['ymin']
lkg_ymax = leakage['ymax']


if rank == 0:
    plt.figure(figsize=(16,9))
    plt.loglog(erg, lkg_xmin, "--", drawstyle='steps', label="XMin")
    plt.loglog(erg, lkg_xmax, "--", drawstyle='steps', label="XMax")
    plt.loglog(erg, lkg_ymin, "--", drawstyle='steps', label="YMin")
    plt.loglog(erg, lkg_ymax, "--", drawstyle='steps', label="YMax")
    plt.xlabel("Energy (MeV)")
    plt.ylabel(r"$\vec{j}^{g}|_{\Gamma_{\text{+}}}$")
    plt.legend()
    plt.grid()
    plt.savefig("Lkg.png")

    print("Total Leakage XMin: ", np.sum(lkg_xmin),"\nTotal Leakage XMax: ",np.sum(lkg_xmax))
    print("Total Leakage YMin: ", np.sum(lkg_ymin),"\nTotal Leakage YMax: ",np.sum(lkg_ymax))
    print("Total Leakage : ", np.sum(lkg_xmin)+np.sum(lkg_xmax)+ \
                              np.sum(lkg_ymin)+np.sum(lkg_ymax))


Total Leakage XMin:  117.534615 \
Total Leakage XMax:  117.538656 \
Total Leakage YMin:  317.871536 \
Total Leakage YMax:  317.870310 

Total Leakage :  870.815119

![Leakage](images/Lkg.png)

### Compute Balance

In [ ]:
balance = phys.ComputeBalance()

## Finalize (for Jupyter Notebook only)

In Python script mode, PyOpenSn automatically handles environment termination. However, this
automatic finalization does not occur when running in a Jupyter notebook, so explicit finalization
of the environment at the end of the notebook is required. Do not call the finalization in Python
script mode, or in console mode.

Note that PyOpenSn's finalization must be called before MPI's finalization.


In [ ]:
from IPython import get_ipython

def finalize_env():
    Finalize()
    MPI.Finalize()

ipython_instance = get_ipython()
if ipython_instance is not None:
    ipython_instance.events.register("post_execute", finalize_env)

os.system("rm -rf Data Flux Results")
